In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
tf_path = "drive/MyDrive/train_ds"
stats_path = "drive/MyDrive/train_ds/stats.npz"

In [3]:
import os
from functools import partial

import numpy as np
import tensorflow as tf

# ==========================================================
# CONFIG
# ==========================================================

# Multi-resolution STFT configs: list of (frame_length, frame_step)
# First entry is the reference resolution used for mask computation
MR_STFT_CONFIGS = [
    (4096, 1024),
    (2048, 512),   # reference: high frequency resolution
    (1024, 256),   # high time resolution
]
CONV_SIZE = 16
EPS = 1e-4


def _make_window_fn(frame_length):
    window = tf.sqrt(tf.signal.hann_window(frame_length, periodic=True))
    def _fn(fl, dtype):
        return tf.cast(window, dtype)
    return _fn


# ==========================================================
# TFRecord schema
# ==========================================================

FEATURE_DESCRIPTION = {
    "length": tf.io.FixedLenFeature([], tf.int64),

    "mix": tf.io.FixedLenFeature([], tf.string),
    "drums": tf.io.FixedLenFeature([], tf.string),
    "bass": tf.io.FixedLenFeature([], tf.string),
    "other": tf.io.FixedLenFeature([], tf.string),
    "vocals": tf.io.FixedLenFeature([], tf.string),
}


# ==========================================================
# Parse TFRecord
# ==========================================================

def parse_tfrecord(example_proto):
    example = tf.io.parse_single_example(
        example_proto,
        FEATURE_DESCRIPTION
    )

    length = tf.cast(example["length"], tf.int32)

    def decode(key):
        x = tf.io.decode_raw(example[key], tf.float32)
        x = tf.reshape(x, [length])
        return x

    return {
        "mix": decode("mix"),
        "drums": decode("drums"),
        "bass": decode("bass"),
        "other": decode("other"),
        "vocals": decode("vocals"),
    }


# ==========================================================
# Padding
# ==========================================================

def pad_feature(x):
    h = tf.shape(x)[0]
    w = tf.shape(x)[1]

    pad_h = (CONV_SIZE - h % CONV_SIZE) % CONV_SIZE
    pad_w = (CONV_SIZE - w % CONV_SIZE) % CONV_SIZE

    paddings = [[0, pad_h], [0, pad_w]]

    ndims = x.shape.rank
    if ndims is not None and ndims > 2:
        paddings.extend([[0, 0]] * (ndims - 2))

    return tf.pad(x, paddings)


# ==========================================================
# Magnitude
# ==========================================================

def magnitude(stft):
    return tf.abs(stft)


def log_magnitude(mag):
    return tf.math.log1p(mag)


# ==========================================================
# Spectrogram
# ==========================================================

def waveform_to_features(audio, configs=None):
    if configs is None:
        configs = MR_STFT_CONFIGS

    ref_fl, ref_fs = configs[0]
    ref_wfn = _make_window_fn(ref_fl)
    ref_stft = tf.signal.stft(audio, frame_length=ref_fl, frame_step=ref_fs, fft_length=ref_fl, window_fn=ref_wfn)
    ref_mag = tf.abs(ref_stft)
    ref_T = tf.shape(ref_mag)[0]
    ref_F = tf.shape(ref_mag)[1]

    channels = []
    for i, (fl, fs) in enumerate(configs):
        wfn = _make_window_fn(fl)
        s = tf.signal.stft(audio, frame_length=fl, frame_step=fs, fft_length=fl, window_fn=wfn)
        mag = tf.abs(s)
        log_mag = tf.math.log1p(mag)

        if i == 0:
            channels.append(log_mag)
        else:
            log_mag = log_mag[tf.newaxis, ..., tf.newaxis]
            log_mag = tf.image.resize(log_mag, [ref_T, ref_F])
            channels.append(log_mag[0, ..., 0])

    feat = tf.stack(channels, axis=-1)
    feat = pad_feature(feat)
    return feat


def pad_batch(x):
    h = tf.shape(x)[1]
    w = tf.shape(x)[2]

    pad_h = (CONV_SIZE - h % CONV_SIZE) % CONV_SIZE
    pad_w = (CONV_SIZE - w % CONV_SIZE) % CONV_SIZE

    return tf.pad(
        x,
        paddings=[
            [0, 0],      # batch
            [0, pad_h],  # height (time frames)
            [0, pad_w]   # width (frequency bins)
        ]
    )

# ==========================================================
# Example preprocessing
# ==========================================================

def preprocess(example, configs=None, augment=False):
    if configs is None:
        configs = MR_STFT_CONFIGS

    tracks = tf.stack([
        example["mix"],
        example["drums"],
        example["bass"],
        example["other"],
        example["vocals"],
    ])

    mix = example["mix"]

    if augment:
        gain = tf.random.uniform([], 0.8, 1.25)
        tracks = tracks * gain
        mix = mix * gain

    # Reference STFT (first config) — used for mask computation
    ref_fl, ref_fs = configs[0]
    ref_wfn = _make_window_fn(ref_fl)
    ref_stft = tf.signal.stft(tracks, frame_length=ref_fl, frame_step=ref_fs, fft_length=ref_fl, window_fn=ref_wfn)
    ref_mag = tf.abs(ref_stft)
    ref_T = tf.shape(ref_mag)[1]
    ref_F = tf.shape(ref_mag)[2]

    # Multi-resolution STFT for mix → input channels
    mix_channels = []
    for i, (fl, fs) in enumerate(configs):
        wfn = _make_window_fn(fl)
        s = tf.signal.stft(mix, frame_length=fl, frame_step=fs, fft_length=fl, window_fn=wfn)
        mag = tf.abs(s)
        log_mag = tf.math.log1p(mag)

        if i == 0:
            mix_channels.append(log_mag)
        else:
            log_mag = log_mag[tf.newaxis, ..., tf.newaxis]
            log_mag = tf.image.resize(log_mag, [ref_T, ref_F])
            mix_channels.append(log_mag[0, ..., 0])

    mix_input = tf.stack(mix_channels, axis=-1)

    # Masks from reference STFT
    mix_mag_ref = ref_mag[0]
    mix_phase = tf.math.angle(ref_stft[0])
    masks = ref_mag[1:] / (ref_mag[0] + EPS)
    masks = tf.clip_by_value(masks, 0.0, 1.0)
    masks = tf.transpose(masks, [1, 2, 0])

    # Pad to CONV_SIZE alignment
    mix_input = pad_feature(mix_input)
    masks = pad_feature(masks)
    mix_mag_ref = pad_feature(mix_mag_ref)
    mix_phase = pad_feature(mix_phase[..., tf.newaxis])

    target = tf.concat([masks, mix_mag_ref[..., tf.newaxis], mix_phase], axis=-1)

    return mix_input, target


# ==========================================================
# Statistics
# ==========================================================

class Normalizer:
    def __init__(self, stats_file=None):
        if stats_file is None:
            self.mean = None
            self.std = None

        else:
            stats = np.load(stats_file)
            mean = stats["mean"]
            std = stats["std"]

            if mean.ndim == 0:
                self.mean = tf.constant(float(mean), dtype=tf.float32)
                self.std = tf.constant(float(std), dtype=tf.float32)
            else:
                self.mean = tf.constant(mean, dtype=tf.float32)
                self.std = tf.constant(std, dtype=tf.float32)

    def __call__(self, x, y):
        if self.mean is None:
            return x, y

        x = (x - self.mean) / (self.std + 1e-8)

        return x, y


def create_dataset(
    tfrecord_dir,
    batch_size,
    stats_file=None,
    shuffle=True,
    shuffle_buffer=128,
    compression_type="GZIP",
    num_parallel=2,
    cycle_length=2,
    prefetch_buffer=2,
    stft_configs=None,
    augment=False,
):
    files=tf.data.Dataset.list_files(
        os.path.join(tfrecord_dir,"*.tfrecord"),
        shuffle=shuffle
    )

    dataset=files.interleave(
        lambda x: tf.data.TFRecordDataset(
            x,
            compression_type=compression_type
        ),
        cycle_length=cycle_length,
        block_length=1,
        num_parallel_calls=num_parallel,
        deterministic=not shuffle
    )

    if shuffle:
        dataset=dataset.shuffle(shuffle_buffer, reshuffle_each_iteration=True)

    dataset=dataset.map(parse_tfrecord, num_parallel_calls=num_parallel)

    dataset=dataset.map(
        partial(preprocess, configs=stft_configs, augment=augment),
        num_parallel_calls=num_parallel,
    )
    normalizer=Normalizer(stats_file)
    dataset=dataset.map(normalizer, num_parallel_calls=num_parallel)
    dataset=dataset.batch(batch_size, drop_remainder=False)
    dataset=dataset.prefetch(prefetch_buffer)

    return dataset

In [20]:
import tensorflow as tf


def _sqrt_hann_window(frame_length):
    window = tf.sqrt(tf.signal.hann_window(frame_length, periodic=True))
    def _fn(fl, dtype):
        return tf.cast(window, dtype)
    return _fn


class SeparatorLoss(tf.keras.losses.Loss):
    def __init__(
        self,
        alpha=1.0,
        beta=1.0,
        gamma=0.1,
        scales=(1, 2, 4),
        ref_frame_length=4096,
        ref_frame_step=1024,
        name="separator_loss",
    ):
        super().__init__(name=name)
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.scales = scales
        self.ref_fl = ref_frame_length
        self.ref_fs = ref_frame_step
        self.ref_wfn = _sqrt_hann_window(ref_frame_length)

    def call(self, y_true, y_pred):
        target_masks = y_true[..., :4]
        mix_mag = y_true[..., 4:5]
        mix_phase = y_true[..., 5:6]

        mask_loss = tf.reduce_mean(tf.abs(target_masks - y_pred))

        spec_loss = 0.0
        for s in self.scales:
            pred_src = mix_mag * y_pred
            target_src = mix_mag * target_masks
            if s > 1:
                pred_src = tf.nn.avg_pool2d(pred_src, s, s, "SAME")
                target_src = tf.nn.avg_pool2d(target_src, s, s, "SAME")
            spec_loss += tf.reduce_mean(tf.abs(pred_src - target_src))
        spec_loss /= len(self.scales)

        return self.alpha * mask_loss + self.beta * spec_loss


CombinedSpectrogramLoss = SeparatorLoss


In [21]:
import tensorflow as tf
from tensorflow.keras import layers, models


def conv_block(x, filters, kernel_size=3):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.GroupNormalization(groups=8)(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.GroupNormalization(groups=8)(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same")(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x


def se_block(x, ratio=8):
    channels = x.shape[-1]
    squeeze = layers.GlobalAveragePooling2D()(x)
    squeeze = layers.Dense(max(channels // ratio, 4), activation="relu")(squeeze)
    squeeze = layers.Dense(channels, activation="sigmoid")(squeeze)
    return layers.multiply([x, squeeze])


class SelfAttentionBlock(layers.Layer):
    def __init__(self, n_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.n_heads = n_heads

    def build(self, input_shape):
        c = input_shape[-1]
        key_dim = max(c // self.n_heads, 8)
        self.mha = layers.MultiHeadAttention(
            num_heads=self.n_heads, key_dim=key_dim
        )
        self.proj = layers.Dense(c)
        self.ln = layers.LayerNormalization()

    def call(self, x):
        b = tf.shape(x)[0]
        h = tf.shape(x)[1]
        w = tf.shape(x)[2]
        c = x.shape[-1]

        x_t = tf.reshape(x, [b * w, h, c])
        attn_out = self.mha(x_t, x_t)
        attn_out = self.proj(attn_out)
        attn_out = tf.reshape(attn_out, [b, w, h, c])
        attn_out = tf.transpose(attn_out, [0, 2, 1, 3])

        return self.ln(x + attn_out)


def create_unet(
    input_shape=(None, None, 1),
    n_filters=16,
    n_levels=3,
    n_outputs=4,
    dropout_rate=0.3,
    use_se=True,
    se_ratio=8,
    use_self_attn=True,
    self_attn_heads=2,
    n_input_channels=None,
):
    if n_input_channels is not None:
        input_shape = (input_shape[0], input_shape[1], n_input_channels)
    inputs = layers.Input(shape=input_shape)

    x = inputs
    skips = []

    for i in range(n_levels):
        filters = n_filters * (2 ** i)
        x = conv_block(x, filters)
        x = conv_block(x, filters)
        if use_se:
            x = se_block(x, se_ratio)
        if use_self_attn and i > 0:
            x = SelfAttentionBlock(n_heads=self_attn_heads)(x)
        skips.append(x)
        x = layers.MaxPooling2D((2, 2))(x)

    filters = n_filters * (2 ** n_levels)
    x = conv_block(x, filters)
    x = conv_block(x, filters)
    if use_se:
        x = se_block(x, se_ratio)
    if use_self_attn:
        x = SelfAttentionBlock(n_heads=self_attn_heads)(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    for i in range(n_levels - 1, -1, -1):
        filters = n_filters * (2 ** i)
        x = layers.Conv2DTranspose(filters, 2, strides=(2, 2), padding="same")(x)
        x = layers.GroupNormalization(groups=8)(x)
        x = layers.Activation("relu")(x)
        x = layers.concatenate([x, skips[i]])
        x = conv_block(x, filters)
        x = conv_block(x, filters)
        if use_se:
            x = se_block(x, se_ratio)
        if use_self_attn and i > 0:
            x = SelfAttentionBlock(n_heads=self_attn_heads)(x)
        if dropout_rate > 0 and i > 0:
            x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Conv2D(n_outputs, 1, padding="same", activation="sigmoid")(x)

    return models.Model(inputs, outputs)


In [24]:
dataset = create_dataset(
    tfrecord_dir=tf_path,
    batch_size=1,
    stats_file=stats_path,  # или None
    shuffle=True,
    augment=True,
    shuffle_buffer=64,
)

In [25]:
class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [tf.Variable(tf.identity(w), trainable=False) for w in self.model.trainable_weights]

    def on_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i].assign(
                self.decay * self.ema_weights[i] + (1 - self.decay) * w
            )

    def on_train_end(self, logs=None):
        for w, ema in zip(self.model.trainable_weights, self.ema_weights):
            w.assign(ema)


In [26]:
def mask_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true[..., :4] - y_pred))

callbacks = [
    EMACallback(),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss", factor=0.5, patience=5, min_lr=1e-7
    ),
]

n_channels = len(MR_STFT_CONFIGS)
model = create_unet(input_shape=(None, None, n_channels))

In [9]:
# import tensorflow
# tensorflow.keras.utils.plot_model(
#     model,
#     to_file="model_architecture.png",
#     show_shapes=True,
#     show_dtype=False,
#     show_layer_names=True,
#     rankdir="TB",
#     expand_nested=True,
#     dpi=200,
#     show_layer_activations=True
# )

In [27]:
LEARNING_RATE = 1e-4

ref_fl, ref_fs = MR_STFT_CONFIGS[0]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=CombinedSpectrogramLoss(alpha=1.0, beta=1.0, gamma=0.1, ref_frame_length=ref_fl, ref_frame_step=ref_fs),
    metrics=[mask_mae],
)

In [28]:
history = model.fit(
    dataset,
    epochs=8,
    callbacks=callbacks,
)

Epoch 1/8


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/separator_loss/inverse_stft/irfft/mul_2 defined at (most recent call last):
<stack traces unavailable>
Incompatible shapes: [4,432,2049] vs. [2064]
	 [[{{node gradient_tape/compile_loss/separator_loss/inverse_stft/irfft/mul_2}}]]
	tf2xla conversion failed while converting __inference_one_step_on_data_123879[]. Run with TF_DUMP_GRAPH_PREFIX=/path/to/dump/dir and --vmodule=xla_compiler=2 to obtain a dump of the compiled functions.
	 [[StatefulPartitionedCall]] [Op:__inference_multi_step_on_iterator_125272]

In [ ]:
import os
import numpy as np
import soundfile as sf
import tensorflow as tf

CONV_SIZE = 16

CHUNK_FRAMES = 432  # ~10s at 44100 Hz / 1024 stride
CHUNK_HOP = CHUNK_FRAMES // 2


def _make_window_fn(frame_length):
    window = tf.sqrt(tf.signal.hann_window(frame_length, periodic=True))
    def _fn(fl, dtype):
        return tf.cast(window, dtype)
    return _fn


def load_audio(path, sr=44100):
    audio, orig_sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if orig_sr != sr:
        audio = tf.signal.resample(audio, orig_sr, sr).numpy()
    return audio


def pad_spectrogram(spec):
    h, w = spec.shape[:2]
    pad_h = (CONV_SIZE - h % CONV_SIZE) % CONV_SIZE
    pad_w = (CONV_SIZE - w % CONV_SIZE) % CONV_SIZE
    if pad_h == 0 and pad_w == 0:
        return spec
    pad_width = [(0, pad_h), (0, pad_w)]
    for _ in range(spec.ndim - 2):
        pad_width.append((0, 0))
    return np.pad(spec, pad_width, mode="constant")


def separate(audio, model, configs=None, stats_file=None):
    if configs is None:
        configs = MR_STFT_CONFIGS

    if stats_file is not None:
        stats = np.load(stats_file)
        norm_mean = stats["mean"].astype(np.float32)
        norm_std = stats["std"].astype(np.float32)
    else:
        norm_mean = norm_std = None

    window_fns = [_make_window_fn(fl) for fl, _ in configs]

    # Reference STFT config (first) — used for phase and ISTFT
    ref_fl, ref_fs = configs[0]
    ref_wfn = window_fns[0]

    ref_stft = tf.signal.stft(audio, frame_length=ref_fl, frame_step=ref_fs, fft_length=ref_fl, window_fn=ref_wfn)
    ref_mag = tf.abs(ref_stft).numpy()
    ref_phase = tf.math.angle(ref_stft).numpy()
    T, F = ref_mag.shape

    # Build multi-resolution input stack
    channels = []
    for i, ((fl, fs), wfn) in enumerate(zip(configs, window_fns)):
        s = tf.signal.stft(audio, frame_length=fl, frame_step=fs, fft_length=fl, window_fn=wfn)
        mag = tf.abs(s).numpy()
        log_mag = np.log1p(mag)

        if i == 0:
            channels.append(log_mag)
        else:
            log_mag = tf.image.resize(log_mag[np.newaxis, :, :, np.newaxis], [T, F]).numpy()
            channels.append(log_mag[0, :, :, 0])

    spec = np.stack(channels, axis=-1).astype(np.float32)  # (T, F, N)

    if norm_mean is not None:
        spec = (spec - norm_mean) / (norm_std + 1e-8)

    padded_spec = pad_spectrogram(spec)
    T_pad, F_pad = padded_spec.shape[:2]

    mask_accum = np.zeros((T_pad, F_pad, 4), dtype=np.float64)
    weight_accum = np.zeros(T_pad, dtype=np.float64)
    ola_window = np.sqrt(np.hanning(CHUNK_FRAMES))

    for start in range(0, T_pad, CHUNK_HOP):
        end = start + CHUNK_FRAMES
        chunk = padded_spec[start:end, :, :]

        pad = 0
        if chunk.shape[0] < CHUNK_FRAMES:
            pad = CHUNK_FRAMES - chunk.shape[0]
            chunk = np.pad(chunk, [(0, pad), (0, 0), (0, 0)])

        chunk_input = chunk[np.newaxis, :, :, :]  # (1, t, f, N)
        masks = model.predict(chunk_input, verbose=0)[0]

        if pad:
            masks = masks[:-pad]

        n = masks.shape[0]
        mask_accum[start : start + n] += masks * ola_window[:n, np.newaxis, np.newaxis]
        weight_accum[start : start + n] += ola_window[:n]

    mask_accum /= weight_accum[:, np.newaxis, np.newaxis] + 1e-8

    masks = mask_accum[:T, :F, :].astype(np.float32)

    # ISTFT reconstruction with reference phase
    sources = []
    for i in range(4):
        source_stft = masks[:, :, i] * ref_mag * np.exp(1j * ref_phase)
        source_wav = tf.signal.inverse_stft(
            source_stft,
            frame_length=ref_fl,
            frame_step=ref_fs,
            fft_length=ref_fl,
            window_fn=ref_wfn,
        ).numpy()
        source_wav = source_wav[: len(audio)]
        sources.append(source_wav)

    return np.stack(sources, axis=-1)


def separate_file(input_path, output_dir, model, stats_file=None):
    audio = load_audio(input_path)
    stems = separate(audio, model, stats_file=stats_file)

    names = ["drums", "bass", "other", "vocals"]
    os.makedirs(output_dir, exist_ok=True)

    base = os.path.splitext(os.path.basename(input_path))[0]

    for i, name in enumerate(names):
        path = os.path.join(output_dir, f"{base}_{name}.wav")
        sf.write(path, stems[:, i], 44100)
        print(f"Saved: {path}")


In [ ]:
# model.save("my_model.keras")

In [ ]:
separate_file("/content/BAD OMENS - Just Pretend.mp3", "out", model, stats_file=stats_path)